# Karabo KNN Categorical Synthetic Data Experiment

This notebook uses the shared `data.csv` file and focuses **only on categorical features**. Numeric feature synthesis is intentionally out of scope. The objective is to generate synthetic categorical customer records using KNN and then evaluate whether the synthetic data preserves the original categorical structure, relationships, rare categories, and downstream target signal.

## 1. Setup and Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Read Shared Data

The notebook expects `data.csv` to be available in the same folder. In this ChatGPT environment, it is available under `/mnt/data/data.csv`.

In [ ]:
DATA_PATH = "/mnt/data/data.csv"  # change to "data.csv" when running locally
df = pd.read_csv(DATA_PATH)
print("Data shape:", df.shape)
display(df.head())
print(df.dtypes)

## 3. Select Categorical Features Only

This experiment excludes numerical columns from synthesis. The target column is retained only for stratification and validation.

In [ ]:
TARGET_COL = "target"

# Keep only categorical/object columns, plus target.
categorical_cols = [
    c for c in df.columns
    if c != TARGET_COL and (df[c].dtype == "object" or str(df[c].dtype) == "category")
]

print(f"Categorical columns selected: {len(categorical_cols)}")
print(categorical_cols)

work_df = df[categorical_cols + [TARGET_COL]].copy()

# Replace missing category values with an explicit bucket.
for col in categorical_cols:
    work_df[col] = work_df[col].astype("object").where(work_df[col].notna(), "__MISSING__").astype(str)

print("Working data shape:", work_df.shape)
display(work_df.head())
print("Target distribution:")
print(work_df[TARGET_COL].value_counts(normalize=True))

## 4. Optional Sampling

Use a sample for faster experiments. Set `SAMPLE_SIZE = None` to use the full dataset.

In [ ]:
SAMPLE_SIZE = 10000

if SAMPLE_SIZE is not None and len(work_df) > SAMPLE_SIZE:
    work_df = work_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print("Experiment data shape:", work_df.shape)

## 5. Encode Categorical Data

KNN needs numeric representation. Ordinal encoding is used only as an internal representation for Hamming distance. We decode the synthetic data back to original category labels later.

In [ ]:
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoded_features = encoder.fit_transform(work_df[categorical_cols])
encoded_df = pd.DataFrame(encoded_features, columns=categorical_cols)
encoded_df[TARGET_COL] = work_df[TARGET_COL].values

display(encoded_df.head())

## 6. Train / Validation Split

In [ ]:
train_df, val_df = train_test_split(
    encoded_df,
    test_size=0.30,
    stratify=encoded_df[TARGET_COL],
    random_state=RANDOM_STATE
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

## 7. KNN Categorical Synthesis Function

Supported methods:

- `mode`: majority value from nearest neighbours. Stable but may reduce diversity.
- `weighted_mode`: closer neighbours get more weight.
- `probability_sampling`: randomly samples neighbour values based on frequency/weight. More diverse.

In [ ]:
def knn_categorical_synthesize(
    train_df,
    feature_cols,
    target_col="target",
    n_samples=5000,
    k=5,
    metric="hamming",
    method="weighted_mode",
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    X = train_df[feature_cols].values

    nn = NearestNeighbors(n_neighbors=k, metric=metric)
    nn.fit(X)

    synth_rows = []

    for _ in range(n_samples):
        base_idx = int(rng.integers(0, len(train_df)))
        distances, indices = nn.kneighbors([X[base_idx]], return_distance=True)
        distances = distances[0]
        indices = indices[0]
        neighbor_df = train_df.iloc[indices]

        # Higher weight for closer neighbours.
        weights = 1 / (distances + 1e-6)
        weights = weights / weights.sum()

        row = {}
        for col in feature_cols:
            vals = neighbor_df[col].values

            if method == "mode":
                row[col] = pd.Series(vals).mode().iloc[0]

            elif method == "weighted_mode":
                score = {}
                for v, w in zip(vals, weights):
                    score[v] = score.get(v, 0) + w
                row[col] = max(score.items(), key=lambda x: x[1])[0]

            elif method == "probability_sampling":
                unique_vals = np.array(list(dict.fromkeys(vals)))
                prob = []
                for uv in unique_vals:
                    prob.append(weights[vals == uv].sum())
                prob = np.array(prob) / np.sum(prob)
                row[col] = rng.choice(unique_vals, p=prob)

            else:
                raise ValueError("method must be one of: mode, weighted_mode, probability_sampling")

        # Assign target by majority vote among nearest neighbours.
        row[target_col] = neighbor_df[target_col].mode().iloc[0]
        synth_rows.append(row)

    return pd.DataFrame(synth_rows)

## 8. Generate Synthetic Data

In [ ]:
N_SYNTHETIC = 5000
K = 5
METHOD = "weighted_mode"  # try: "mode", "weighted_mode", "probability_sampling"

synth_encoded = knn_categorical_synthesize(
    train_df=train_df,
    feature_cols=categorical_cols,
    target_col=TARGET_COL,
    n_samples=N_SYNTHETIC,
    k=K,
    method=METHOD,
    random_state=RANDOM_STATE,
)

print("Synthetic encoded shape:", synth_encoded.shape)
display(synth_encoded.head())

## 9. Decode Synthetic Data Back to Category Labels

In [ ]:
synth_decoded = pd.DataFrame(
    encoder.inverse_transform(synth_encoded[categorical_cols]),
    columns=categorical_cols
)
synth_decoded[TARGET_COL] = synth_encoded[TARGET_COL].values

train_decoded = pd.DataFrame(
    encoder.inverse_transform(train_df[categorical_cols]),
    columns=categorical_cols,
    index=train_df.index
)
train_decoded[TARGET_COL] = train_df[TARGET_COL].values

display(synth_decoded.head())

## 10. Evaluation Functions

In [ ]:
def distribution_drift(original_df, synthetic_df, cols):
    drift = {}
    for col in cols:
        orig = original_df[col].value_counts(normalize=True)
        synth = synthetic_df[col].value_counts(normalize=True)
        cats = orig.index.union(synth.index)
        drift[col] = (orig.reindex(cats, fill_value=0) - synth.reindex(cats, fill_value=0)).abs().mean()
    return drift

def rare_category_retention(original_df, synthetic_df, cols, threshold=0.05):
    results = {}
    for col in cols:
        orig_freq = original_df[col].value_counts(normalize=True)
        rare_cats = orig_freq[orig_freq < threshold].index
        if len(rare_cats) == 0:
            results[col] = np.nan
            continue
        synth_freq = synthetic_df[col].value_counts(normalize=True)
        orig_mass = orig_freq.loc[rare_cats].sum()
        synth_mass = synth_freq.reindex(rare_cats, fill_value=0).sum()
        results[col] = synth_mass / orig_mass if orig_mass > 0 else np.nan
    return results

def cramers_v_from_series(x, y):
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = stats.chi2_contingency(table)[0]
    n = table.sum().sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

def relationship_delta(original_df, synthetic_df, cols, max_pairs=200):
    pairs = []
    for i, c1 in enumerate(cols):
        for c2 in cols[i+1:]:
            pairs.append((c1, c2))
    pairs = pairs[:max_pairs]

    deltas = []
    for c1, c2 in pairs:
        ov = cramers_v_from_series(original_df[c1], original_df[c2])
        sv = cramers_v_from_series(synthetic_df[c1], synthetic_df[c2])
        if not np.isnan(ov) and not np.isnan(sv):
            deltas.append(abs(ov - sv))
    return float(np.mean(deltas)) if deltas else np.nan

def calc_iv(df, feature, target):
    eps = 1e-6
    total_good = (df[target] == 0).sum()
    total_bad = (df[target] == 1).sum()
    iv = 0
    for val, sub in df.groupby(feature):
        good = (sub[target] == 0).sum()
        bad = (sub[target] == 1).sum()
        dist_good = good / max(total_good, 1)
        dist_bad = bad / max(total_bad, 1)
        woe = np.log((dist_good + eps) / (dist_bad + eps))
        iv += (dist_good - dist_bad) * woe
    return iv

def iv_retention(original_df, synthetic_df, cols, target):
    rows = []
    for col in cols:
        orig_iv = calc_iv(original_df, col, target)
        synth_iv = calc_iv(synthetic_df, col, target)
        rows.append({
            "feature": col,
            "original_iv": orig_iv,
            "synthetic_iv": synth_iv,
            "retention_ratio": synth_iv / orig_iv if orig_iv > 1e-9 else np.nan,
            "absolute_delta": abs(orig_iv - synth_iv),
        })
    return pd.DataFrame(rows).sort_values("absolute_delta", ascending=False)

def duplicate_report(synthetic_df):
    return {
        "rows": len(synthetic_df),
        "unique_rows": len(synthetic_df.drop_duplicates()),
        "duplicate_rate": float(synthetic_df.duplicated().mean()),
    }

## 11. Run Evaluation

In [ ]:
drift = distribution_drift(train_decoded, synth_decoded, categorical_cols)
rare = rare_category_retention(train_decoded, synth_decoded, categorical_cols)
iv_df = iv_retention(train_decoded, synth_decoded, categorical_cols, TARGET_COL)
rel_delta = relationship_delta(train_decoded, synth_decoded, categorical_cols)
dup = duplicate_report(synth_decoded)

scorecard = pd.DataFrame([
    {"metric": "mean_distribution_drift", "value": np.nanmean(list(drift.values()))},
    {"metric": "mean_rare_category_retention", "value": np.nanmean(list(rare.values()))},
    {"metric": "mean_cramers_v_delta", "value": rel_delta},
    {"metric": "duplicate_rate", "value": dup["duplicate_rate"]},
    {"metric": "target_rate_original_train", "value": train_decoded[TARGET_COL].mean()},
    {"metric": "target_rate_synthetic", "value": synth_decoded[TARGET_COL].mean()},
])

display(scorecard)
display(iv_df.head(15))

## 12. Visual Checks

In [ ]:
top_cols = pd.Series(drift).sort_values(ascending=False).head(8).index.tolist()

for col in top_cols:
    orig = train_decoded[col].value_counts(normalize=True).head(10)
    synth = synth_decoded[col].value_counts(normalize=True).reindex(orig.index, fill_value=0)
    comp = pd.DataFrame({"original": orig, "synthetic": synth})
    ax = comp.plot(kind="bar", figsize=(10, 4), title=f"Distribution comparison: {col}")
    ax.set_ylabel("frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 13. Method and K Sensitivity Test

In [ ]:
sensitivity_rows = []

for method in ["mode", "weighted_mode", "probability_sampling"]:
    for k in [3, 5, 10, 20]:
        tmp_enc = knn_categorical_synthesize(
            train_df=train_df,
            feature_cols=categorical_cols,
            target_col=TARGET_COL,
            n_samples=2000,
            k=k,
            method=method,
            random_state=RANDOM_STATE,
        )
        tmp_dec = pd.DataFrame(encoder.inverse_transform(tmp_enc[categorical_cols]), columns=categorical_cols)
        tmp_dec[TARGET_COL] = tmp_enc[TARGET_COL].values
        tmp_drift = distribution_drift(train_decoded, tmp_dec, categorical_cols)
        tmp_rare = rare_category_retention(train_decoded, tmp_dec, categorical_cols)
        tmp_dup = duplicate_report(tmp_dec)
        sensitivity_rows.append({
            "method": method,
            "k": k,
            "mean_distribution_drift": np.nanmean(list(tmp_drift.values())),
            "mean_rare_retention": np.nanmean(list(tmp_rare.values())),
            "duplicate_rate": tmp_dup["duplicate_rate"],
            "target_rate": tmp_dec[TARGET_COL].mean(),
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df.sort_values(["mean_distribution_drift", "duplicate_rate"]))

## 14. Export Outputs

In [ ]:
synth_decoded.to_csv("synthetic_categorical_data.csv", index=False)
scorecard.to_csv("synthetic_categorical_scorecard.csv", index=False)
iv_df.to_csv("synthetic_categorical_iv_retention.csv", index=False)

print("Saved outputs:")
print("- synthetic_categorical_data.csv")
print("- synthetic_categorical_scorecard.csv")
print("- synthetic_categorical_iv_retention.csv")

## 15. Suggested Next Steps

1. Compare K values and generation methods.
2. Add duplicate and privacy leakage checks before sharing synthetic data.
3. Add rare-category protection for important but low-frequency categories.
4. Add business rules for invalid category combinations.
5. Use probability sampling when diversity is more important than strict stability.
6. Use weighted mode when local consistency is more important than diversity.